# Phase 1 : Setup & Literature Foundation

In [ ]:
# CELL 1 — RUN THIS EVERY TIME YOU OPEN COLAB

from google.colab import drive
drive.mount('/content/drive')
!pip install wfdb
import os, ast, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import wfdb

warnings.filterwarnings('ignore')

# Project paths
PROJECT_PATH = '/content/drive/MyDrive/FairWear_Project/'
DATA_PATH    = PROJECT_PATH + 'data/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1/'

print("✓ Drive mounted")
print("✓ Libraries loaded")
print("✓ Paths set")
print(f"\n  PROJECT : {PROJECT_PATH}")
print(f"  DATA    : {DATA_PATH}")

In [ ]:
# CELL 2 — RELOAD DATA (every session)

# Load saved processed dataframe
df = pd.read_csv(PROJECT_PATH + 'ptbxl_processed.csv', index_col='ecg_id')

# Recreate derived columns lost during CSV save
df['age_group'] = pd.cut(df['age'],
                          bins=[0, 30, 45, 60, 75, 120],
                          labels=['<30', '30-45', '45-60', '60-75', '75+'])

df['label']  = df['label'].astype(int)
df['sex']    = df['sex'].astype(int)

# Restore scp_codes from string back to dict
df['scp_codes'] = df['scp_codes'].apply(ast.literal_eval)

print("✓ Data reloaded successfully")
print(f"\n  Rows          : {len(df)}")
print(f"  Unique patients: {df['patient_id'].nunique()}")
print(f"  Male / Female  : {(df['sex']==0).sum()} / {(df['sex']==1).sum()}")
print(f"  Normal         : {(df['label']==0).sum()}")
print(f"  Abnormal       : {(df['label']==1).sum()}")
print(f"  Age groups     : {df['age_group'].value_counts().sort_index().to_dict()}")
print(f"  Missing age    : {df['age'].isna().sum()}")
print("\n✓ Ready to continue!")

Installing dependencies

In [ ]:
!pip install wfdb neurokit2 imbalanced-learn scikit-learn seaborn matplotlib pandas numpy tensorflow

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("Sklearn:", sklearn.__version__)
print("All libraries loaded successfully ✓")

# Phase 2: Dataset Acquisition & EDA

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/FairWear_Project', exist_ok=True)
os.makedirs('/content/drive/MyDrive/FairWear_Project/data', exist_ok=True)

print("Drive mounted and folders created ✓")
print("Your project folder: /content/drive/MyDrive/FairWear_Project/")

Using kaggle API

In [ ]:
import json, os

kaggle_username = input("Enter your Kaggle username: ")
kaggle_api_key = input("Enter your Kaggle API key: ")

kaggle_config = {
    "username": kaggle_username,
    "key": kaggle_api_key
}

os.makedirs('/root/.config/kaggle', exist_ok=True)
with open('/root/.config/kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_config, f)
os.chmod('/root/.config/kaggle/kaggle.json', 0o600)
print("kaggle.json created successfully ✓")
print("Config saved at: /root/.config/kaggle/kaggle.json")

Download PTB-XL via Kaggle

In [ ]:
!pip install kaggle -q

!kaggle datasets download -d khyeh0719/ptb-xl-dataset \
    -p '/content/drive/MyDrive/FairWear_Project/data/' \
    --unzip

print("Download complete ✓")

In [ ]:
import os

data_path = '/content/drive/MyDrive/FairWear_Project/data/'

print("Files downloaded:")
for f in os.listdir(data_path):
    size = os.path.getsize(os.path.join(data_path, f))
    print(f"  {f}  —  {size/1e6:.1f} MB")

In [ ]:
import os

DATA_PATH = '/content/drive/MyDrive/FairWear_Project/data/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1/'

required_files = ['ptbxl_database.csv', 'scp_statements.csv']
required_folders = ['records100']

print("Checking required files:")
for f in required_files:
    full_path = os.path.join(DATA_PATH, f)
    exists = os.path.exists(full_path)
    print(f"  {'✓' if exists else '✗'} {f}")

print("\nChecking required folders:")
for folder in required_folders:
    full_path = os.path.join(DATA_PATH, folder)
    exists = os.path.exists(full_path)
    print(f"  {'✓' if exists else '✗'} {folder}")

print("\nAll contents:")
for item in os.listdir(DATA_PATH):
    print(f"  {item}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast

df = pd.read_csv(DATA_PATH + 'ptbxl_database.csv', index_col='ecg_id')
df['scp_codes'] = df['scp_codes'].apply(ast.literal_eval)

print(f"Dataset loaded ✓")
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")

In [ ]:
# ── Demographics ──

# Age groups
df['age_group'] = pd.cut(df['age'],
                          bins=[0, 30, 45, 60, 75, 120],
                          labels=['<30', '30-45', '45-60', '60-75', '75+'])

# Gender counts (0=Male, 1=Female in PTB-XL)
gender_counts = df['sex'].value_counts()
gender_pct = df['sex'].value_counts(normalize=True) * 100

print("="*50)
print("DEMOGRAPHIC AUDIT SUMMARY")
print("="*50)
print(f"\nTotal Records : {len(df)}")
print(f"Unique Patients: {df['patient_id'].nunique()}")
print(f"\nMale   : {gender_counts[0]} ({gender_pct[0]:.1f}%)")
print(f"Female : {gender_counts[1]} ({gender_pct[1]:.1f}%)")
print(f"\nMean Age   : {df['age'].mean():.1f} yrs")
print(f"Median Age : {df['age'].median():.1f} yrs")
print(f"Age Range  : {df['age'].min():.0f} – {df['age'].max():.0f} yrs")
print(f"Missing Age: {df['age'].isna().sum()}")

print("\nAge Group Distribution:")
print(df['age_group'].value_counts().sort_index())

In [ ]:
# ── Diagnostic Labels ──

agg_df = pd.read_csv(DATA_PATH + 'scp_statements.csv', index_col=0)
agg_df = agg_df[agg_df.diagnostic == 1]

def aggregate_diagnostic(y_dic):
    tmp = []
    for key in y_dic.keys():
        if key in agg_df.index:
            tmp.append(agg_df.loc[key].diagnostic_class)
    return list(set(tmp))

def is_abnormal(classes):
    if 'NORM' in classes and len(classes) == 1:
        return 0
    elif len(classes) > 0:
        return 1
    return np.nan

df['diagnostic_superclass'] = df['scp_codes'].apply(aggregate_diagnostic)
df['label'] = df['diagnostic_superclass'].apply(is_abnormal)
df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)

print("Label Distribution:")
print(df['label'].value_counts())
print("0 = Normal | 1 = Abnormal")

In [ ]:
# ── All EDA Plots ──

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('PTB-XL Dataset — Demographic Audit\nFairWear Auditor',
             fontsize=15, fontweight='bold')

# 1: Gender Pie
axes[0,0].pie([gender_counts[0], gender_counts[1]],
              labels=['Male', 'Female'],
              autopct='%1.1f%%',
              colors=['#4C72B0', '#DD8452'])
axes[0,0].set_title('Gender Distribution')

# 2: Age Histogram
axes[0,1].hist(df['age'].dropna(), bins=30, color='#4C72B0',
               edgecolor='white', alpha=0.8)
axes[0,1].axvline(df['age'].mean(), color='red', linestyle='--',
                   label=f"Mean: {df['age'].mean():.1f}")
axes[0,1].set_xlabel('Age (years)')
axes[0,1].set_ylabel('Count')
axes[0,1].set_title('Age Distribution')
axes[0,1].legend()

# 3: Age Groups Bar
age_grp = df['age_group'].value_counts().sort_index()
axes[0,2].bar(age_grp.index, age_grp.values,
              color='#55A868', edgecolor='white')
axes[0,2].set_xlabel('Age Group')
axes[0,2].set_ylabel('Count')
axes[0,2].set_title('Age Group Distribution')

# 4: Label Distribution
label_counts = df['label'].value_counts()
axes[1,0].bar(['Normal', 'Abnormal'], label_counts.values,
              color=['#2ecc71', '#e74c3c'], edgecolor='white')
axes[1,0].set_ylabel('Count')
axes[1,0].set_title('Diagnostic Label Distribution')
for i, v in enumerate(label_counts.values):
    axes[1,0].text(i, v + 100, str(v), ha='center', fontweight='bold')

# 5: Labels by Gender
label_gender = df.groupby(['sex', 'label']).size().unstack()
label_gender.index = ['Male', 'Female']
label_gender.plot(kind='bar', ax=axes[1,1],
                  color=['#2ecc71', '#e74c3c'], edgecolor='white')
axes[1,1].set_xlabel('Gender')
axes[1,1].set_ylabel('Count')
axes[1,1].set_title('Diagnoses by Gender')
axes[1,1].legend(['Normal', 'Abnormal'])
axes[1,1].tick_params(axis='x', rotation=0)

# 6: Age by Label
df[df['label']==0]['age'].hist(ax=axes[1,2], bins=25,
                                alpha=0.6, color='#2ecc71', label='Normal')
df[df['label']==1]['age'].hist(ax=axes[1,2], bins=25,
                                alpha=0.6, color='#e74c3c', label='Abnormal')
axes[1,2].set_xlabel('Age (years)')
axes[1,2].set_ylabel('Count')
axes[1,2].set_title('Age Distribution by Diagnosis')
axes[1,2].legend()

plt.tight_layout()

save_path = '/content/drive/MyDrive/FairWear_Project/demographic_audit.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved to Drive ✓")

In [ ]:
# ── Bias Indicators Table ──

print("="*55)
print("TABLE 1: DATASET BIAS INDICATORS")
print("="*55)

for sex_val, sex_name in [(0, 'Male'), (1, 'Female')]:
    subset = df[df['sex'] == sex_val]
    rate = subset['label'].mean() * 100
    print(f"\n{sex_name}:")
    print(f"  Total records   : {len(subset)}")
    print(f"  Abnormal cases  : {subset['label'].sum()}")
    print(f"  Abnormality rate: {rate:.1f}%")

print("\n--- By Age Group ---")
age_bias = df.groupby('age_group')['label'].agg(['count','sum','mean'])
age_bias.columns = ['Total', 'Abnormal', 'Abn. Rate %']
age_bias['Abn. Rate %'] = (age_bias['Abn. Rate %'] * 100).round(1)
print(age_bias)

df.to_csv('/content/drive/MyDrive/FairWear_Project/ptbxl_processed.csv')
print("\nProcessed dataframe saved to Drive ✓")
print("Ready for Phase 3!")

# Phase 3: Baseline Model

Verify Signal Loading

In [ ]:
sample_path = DATA_PATH + df.iloc[0]['filename_lr']
signal, metadata = wfdb.rdsamp(sample_path)

print("Signal loading test:")
print(f"  Shape         : {signal.shape}")
print(f"  Sampling rate : {metadata['fs']} Hz")
print(f"  Lead names    : {metadata['sig_name']}")
print(f"  Duration      : {signal.shape[0]/metadata['fs']:.1f} seconds")
print(f"\n✓ Signals loading correctly!")

Move Data to Local Storage

In [ ]:
# 1. Local folder on the Colab
!mkdir -p /content/ptbxl_local

# 2. Copy from Drive to local.
DRIVE_PATH = "/content/drive/MyDrive/FairWear_Project/data/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1/"

print("Copying files to local storage (this takes ~2-3 mins)...")
!cp -r "{DRIVE_PATH}." /content/ptbxl_local/

# 3. Update the DATA_PATH to point to the local copy
DATA_PATH = '/content/ptbxl_local/'
print("Data move complete! Now reading from local disk. ✓")

Load ECG Signals

In [ ]:
import numpy as np
import wfdb
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor

def load_single_signal(file_path):
    try:
        signal, _ = wfdb.rdsamp(file_path)
        return signal
    except:
        return None

print("Starting parallel load from local storage...")
file_paths = [DATA_PATH + f for f in df['filename_lr']]

# ThreadPoolExecutor to load files in parallel
with ThreadPoolExecutor(max_workers=4) as executor:
    signals = list(tqdm(executor.map(load_single_signal, file_paths),
                        total=len(file_paths),
                        desc="Loading ECGs"))

# Cleanup: Filter out any records that failed to load
failed_indices = [i for i, s in enumerate(signals) if s is None]
if failed_indices:
    print(f"Removing {len(failed_indices)} failed records...")
    df = df.drop(df.index[failed_indices]).reset_index(drop=True)
    signals = [s for s in signals if s is not None]

# Convert to final Numpy arrays
X_raw = np.array(signals)
y = df['label'].values

print(f"\n✓ Phase 3 Loading Complete!")
print(f"  X shape (Signals) : {X_raw.shape}")
print(f"  y shape (Labels)  : {y.shape}")

Preprocess, Split, and Normalize

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

print("Step 3.2: Starting Preprocessing...")

# 1. Encode string labels to binary
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print(f"Classes mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# 2. Split Data (70% Train, 15% Validation, 15% Test)
# CRITICAL: We split the INDICES so we can isolate the metadata (Age, Gender) for the test set.
indices = np.arange(len(X_raw))

idx_train, idx_temp = train_test_split(indices, test_size=0.30, random_state=42, stratify=y_encoded)
idx_val, idx_test = train_test_split(idx_temp, test_size=0.50, random_state=42, stratify=y_encoded[idx_temp])

X_train, y_train = X_raw[idx_train], y_encoded[idx_train]
X_val,   y_val   = X_raw[idx_val],   y_encoded[idx_val]
X_test,  y_test  = X_raw[idx_test],  y_encoded[idx_test]

# Save the Test set metadata for Phase 4
df_test = df.iloc[idx_test].copy()
df_test['y_true'] = y_test

print(f"\nSplit Sizes:")
print(f"  Training set   : {X_train.shape[0]} samples")
print(f"  Validation set : {X_val.shape[0]} samples")
print(f"  Test set       : {X_test.shape[0]} samples")

# 3. Normalize the Signals
# Neural networks require normalized inputs. We fit the scaler ONLY on the training data.
print("\nNormalizing signals (this might take a few seconds)...")
scaler = StandardScaler()

# Reshape to 2D for scaling, then reshape back to 3D
X_train_flat = X_train.reshape(-1, X_train.shape[-1])
X_train_scaled = scaler.fit_transform(X_train_flat).reshape(X_train.shape)

X_val_flat = X_val.reshape(-1, X_val.shape[-1])
X_val_scaled = scaler.transform(X_val_flat).reshape(X_val.shape)

X_test_flat = X_test.reshape(-1, X_test.shape[-1])
X_test_scaled = scaler.transform(X_test_flat).reshape(X_test.shape)

print("✓ Preprocessing, Splitting, and Normalization Complete!")

Model Architecture and Training

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

print("Step 3.3: Building 1D-CNN Architecture...")

# 1. Define the 1D-CNN Model
def build_baseline_cnn(input_shape):
    model = models.Sequential([
        # First Convolutional Block
        layers.Conv1D(filters=32, kernel_size=5, activation='relu', input_shape=input_shape),
        layers.MaxPooling1D(pool_size=2),
        layers.Dropout(0.2), # Dropout prevents overfitting

        # Second Convolutional Block
        layers.Conv1D(filters=64, kernel_size=5, activation='relu'),
        layers.MaxPooling1D(pool_size=2),
        layers.Dropout(0.2),

        # Flatten and Dense Layers
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),

        # Output Layer: 1 neuron with Sigmoid (Output is probability of being 'Abnormal')
        layers.Dense(1, activation='sigmoid')
    ])
    return model

# Create the model using the shape of our scaled training data
input_shape = (X_train_scaled.shape[1], X_train_scaled.shape[2])
model = build_baseline_cnn(input_shape)

# Compile the model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

model.summary()
print("\n✓ Model Built Successfully!")

print("\nStep 3.4: Starting Training...")

# 2. Train the Model
# We use EarlyStopping so the model stops training if it stops improving, saving time.
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_auc',
    mode='max',
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    X_train_scaled, y_train,
    epochs=15,
    batch_size=64,
    validation_data=(X_val_scaled, y_val),
    callbacks=[early_stopping],
    verbose=1
)

print("\n✓ Baseline Training Complete!")

# 3. Plot Training History
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Model Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['auc'], label='Train AUC')
plt.plot(history.history['val_auc'], label='Val AUC')
plt.title('Model AUC')
plt.legend()
plt.show()

# Phase 4: Bias Quantification

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, roc_auc_score

print("Step 4.1: Generating Predictions on Test Set...")
# 1. Get model probabilities
y_pred_probs = model.predict(X_test_scaled)

# 2. Convert to binary predictions (threshold = 0.5)
y_pred = (y_pred_probs > 0.5).astype(int).flatten()

# 3. Add predictions to our saved test dataframe
df_test['y_pred'] = y_pred
df_test['y_prob'] = y_pred_probs
print("✓ Predictions generated and merged with metadata!\n")

# Helper function to calculate metrics
def calculate_metrics(group_df):
    # Check if group is empty to avoid errors
    if len(group_df) == 0:
        return 0, 0, 0

    tn, fp, fn, tp = confusion_matrix(group_df['y_true'], group_df['y_pred'], labels=[0,1]).ravel()
    acc = accuracy_score(group_df['y_true'], group_df['y_pred'])

    # TPR (Sensitivity/Recall) = TP / (TP + FN)
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
    # FNR (Miss Rate) = FN / (TP + FN) -> We want this to be LOW and EQUAL across groups
    fnr = fn / (tp + fn) if (tp + fn) > 0 else 0

    return round(acc, 3), round(tpr, 3), round(fnr, 3)

print("Step 4.2: Auditing Bias by Gender...")
# Assuming 'sex' column: 0 is Male, 1 is Female (adjust if your EDA mapping was different)
male_df = df_test[df_test['sex'] == 0]
female_df = df_test[df_test['sex'] == 1]

m_acc, m_tpr, m_fnr = calculate_metrics(male_df)
f_acc, f_tpr, f_fnr = calculate_metrics(female_df)

print(f"Male   (n={len(male_df)}) -> Accuracy: {m_acc} | TPR: {m_tpr} | FNR (Miss Rate): {m_fnr}")
print(f"Female (n={len(female_df)}) -> Accuracy: {f_acc} | TPR: {f_tpr} | FNR (Miss Rate): {f_fnr}")
print(f">> Gender FNR Disparity: {abs(m_fnr - f_fnr):.3f} (Ideal is 0.000)\n")

print("Step 4.3: Auditing Bias by Age...")
# Using 60 as a threshold for Younger vs Older
young_df = df_test[df_test['age'] < 60]
old_df = df_test[df_test['age'] >= 60]

y_acc, y_tpr, y_fnr = calculate_metrics(young_df)
o_acc, o_tpr, o_fnr = calculate_metrics(old_df)

print(f"Young (<60)  (n={len(young_df)}) -> Accuracy: {y_acc} | TPR: {y_tpr} | FNR: {y_fnr}")
print(f"Older (>=60) (n={len(old_df)}) -> Accuracy: {o_acc} | TPR: {o_tpr} | FNR: {o_fnr}")
print(f">> Age FNR Disparity: {abs(y_fnr - o_fnr):.3f} (Ideal is 0.000)")

# Phase 5: Lightweight Mitigations

In [ ]:
from sklearn.utils.class_weight import compute_sample_weight

print("Step 5.1: Calculating Subgroup Sample Weights...")
# 1. Reconstruct the training dataframe using the indices from Phase 3
df_train = df.iloc[idx_train].copy()
df_train['y_true'] = y_train

# 2. Create a composite group: TrueLabel + AgeGroup + Gender
# This creates specific buckets like "Abnormal_Young_Female"
composite_groups = (
    df_train['y_true'].astype(str) + "_" +
    (df_train['age'] < 60).astype(str) + "_" +
    df_train['sex'].astype(str)
)

# 3. Compute sample weights (underrepresented groups get higher weights)
sample_weights = compute_sample_weight(class_weight='balanced', y=composite_groups)

print("✓ Weights calculated! Higher weights assigned to minority groups.\n")

print("Step 5.2: Retraining the Mitigated 'Fair' Model...")
# 4. Build a fresh model so it doesn't remember the biased training
fair_model = build_baseline_cnn(input_shape)

fair_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# Train using the sample_weights
fair_history = fair_model.fit(
    X_train_scaled, y_train,
    sample_weight=sample_weights, # THE LIGHTWEIGHT MITIGATION
    epochs=15,
    batch_size=64,
    validation_data=(X_val_scaled, y_val),
    callbacks=[early_stopping],
    verbose=1
)

print("\nStep 5.3: Re-auditing the Fair Model...")
# Get new predictions
y_pred_probs_fair = fair_model.predict(X_test_scaled)
df_test['y_pred'] = (y_pred_probs_fair > 0.5).astype(int).flatten()

# Re-audit Gender
m_acc, m_tpr, m_fnr = calculate_metrics(df_test[df_test['sex'] == 0])
f_acc, f_tpr, f_fnr = calculate_metrics(df_test[df_test['sex'] == 1])
print(f"\n--- MITIGATED GENDER BIAS ---")
print(f"Male   -> FNR: {m_fnr}")
print(f"Female -> FNR: {f_fnr}")
print(f">> New Gender FNR Disparity: {abs(m_fnr - f_fnr):.3f}")

# Re-audit Age
y_acc, y_tpr, y_fnr = calculate_metrics(df_test[df_test['age'] < 60])
o_acc, o_tpr, o_fnr = calculate_metrics(df_test[df_test['age'] >= 60])
print(f"\n--- MITIGATED AGE BIAS ---")
print(f"Young (<60)  -> FNR: {y_fnr}")
print(f"Older (>=60) -> FNR: {o_fnr}")
print(f">> New Age FNR Disparity: {abs(y_fnr - o_fnr):.3f}")